In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import copy
from scipy.stats import wilcoxon, mannwhitneyu
import pickle as pkl

import nibabel as nib

import matplotlib.pyplot as plt 

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import torch
from pytorch_tabnet.tab_model import TabNetClassifier

In [ ]:
def load_unet_result(path, _print):
    unet_df = pd.read_csv(path, index_col = 'Unnamed: 0')
    index_values = []
    for _index in unet_df.index:
        index_values.append(_index.split('-seg')[0]  # strip '-seg' suffix added by U-Net inference script)

    unet_df.index = index_values  
    unet_df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard']  # Dice is the primary metric; Jaccard unused, axis = 1, inplace = True)
    summary_unet_df = pd.DataFrame(zip(unet_df.mean().values.tolist(), 
                                       unet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = unet_df.columns)
    if _print:
        print("****UNet******")
        print(summary_unet_df)
    return summary_unet_df, unet_df

def load_nnunet_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case in data['metric_per_case']:
        WT.append(case['metrics']['(2, 1, 3)']['Dice'])
        TC.append(case['metrics']['(2, 3)']['Dice'])
        ET.append(case['metrics']['(3,)']['Dice'])
        file_name.append(case['reference_file'].split('/')[-1].split('.')[0])

    nnunet_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_nnunet_df = pd.DataFrame(zip(nnunet_df.mean().values.tolist(), 
                                       nnunet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = nnunet_df.columns)
    if _print:
        print("****nnUNet******")
        print(summary_nnunet_df)
    return summary_nnunet_df, nnunet_df

def load_TransBTS_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case_id in data.keys():
        case = data[case_id]
        WT.append(case['WT'][0])
        TC.append(case['TC'][0])
        ET.append(case['ET'][0])
        file_name.append(case_id)

    TransBTS_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_TransBTS_df = pd.DataFrame(zip(TransBTS_df.mean().values.tolist(), 
                                       TransBTS_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = TransBTS_df.columns)
    if _print:
        print("****TransBTS******")
        print(summary_TransBTS_df)
    return summary_TransBTS_df, TransBTS_df

In [ ]:
def read_results(_print=True):
    path = '../Results/Result/Vanilla_Unet/Unet_test_dice.csv'
    summary_unet_df, unet_df = load_unet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainer/summary.json'
    summary_da_nnunet_df, nnunet_da_df = load_nnunet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json'
    summary_noda_nnunet_df, nnunet_noda_df = load_nnunet_result(path, _print)

    path = '../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json'
    summary_TransBTS_df, TransBTS_df = load_TransBTS_result(path, _print)
    return unet_df, nnunet_noda_df, nnunet_da_df, TransBTS_df




In [ ]:
def get_overlaps(unet_df, TransBTS_df, nnunet_noda_df, WT_dice_threshold, TC_dice_threshold, ET_dice_threshold):
    unet_df_sub = unet_df[(unet_df['WT dice'] < WT_dice_threshold) 
                          & (unet_df['TC dice'] < TC_dice_threshold) 
                          & (unet_df['ET dice'] < ET_dice_threshold)]
    
    TransBTS_df_sub = TransBTS_df[(TransBTS_df['WT dice'] < WT_dice_threshold) 
                          & (TransBTS_df['TC dice'] < TC_dice_threshold) 
                          & (TransBTS_df['ET dice'] < ET_dice_threshold)]
    
    nnunet_noda_df_sub = nnunet_noda_df[(nnunet_noda_df['WT dice'] < WT_dice_threshold) 
                          & (nnunet_noda_df['TC dice'] < TC_dice_threshold) 
                          & (nnunet_noda_df['ET dice'] < ET_dice_threshold)]

    unet_df_sub_subjects = unet_df_sub.index.values.tolist()
    TransBTS_df_sub_subjects = TransBTS_df_sub.index.values.tolist()
    nnunet_noda_df_sub_subjects = nnunet_noda_df_sub.index.values.tolist()

    all_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('all overlap', len(all_overlaps), unet_df_sub.shape, nnunet_noda_df_sub.shape)

    unet_nnunet_overlaps = list(set(unet_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('unet-nnunet overlap', len(unet_nnunet_overlaps))

    unet_TransBTS_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects))
    print('unet-TransBTS overlap', len(unet_TransBTS_overlaps))

    nnunet_TransBTS_overlaps = list(set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('nnunet-TransBTS overlap', len(nnunet_TransBTS_overlaps))
    
    return all_overlaps, unet_nnunet_overlaps


In [ ]:
def read_radiomics_results(analysis_type):
    file_name = '../Results/Analysis_Results/Radiomics/Tumor/' + analysis_type + '.pkl'
    with open(file_name, 'rb') as f:
        results = pkl.load(f)
    return results

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = patient_id + '/' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img 

def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def read_mask_file(patient_id):
    baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
    pefix = patient_id + '/' + patient_id
    suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']
    outputloc = '../Results/Result/Vanilla_Unet/' + patient_id
    
    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
    
    masks = preprocess_mask_labels(sample_mask)
    mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]
    
    return mask_WT

def remove_duplicate_columns(df):
    """
    Remove columns from a DataFrame that have identical values to other columns, regardless of the column name.
    
    Parameters:
    - df: The pandas DataFrame from which to remove duplicate columns.
    
    Returns:
    - A new DataFrame with duplicate columns removed.
    """
    columns_to_remove = set()
    for i in range(df.shape[1]): # Iterate over all columns
        for j in range(i + 1, df.shape[1]): # Compare each column with every other column
            # Check if not already identified as duplicate and if equal
            if i not in columns_to_remove and j not in columns_to_remove:
                if df.iloc[:, i].equals(df.iloc[:, j]):
                    columns_to_remove.add(j)
    
    # Create a new DataFrame without the duplicate columns
    df_cleaned = df.drop(columns=df.columns[list(columns_to_remove)])
    return df_cleaned

In [ ]:
# Thresholds from paper: cases below all three are concordant-poor
WT_dice_threshold= 0.91
TC_dice_threshold= 0.86
ET_dice_threshold= 0.85


analysis_types = ['firstorder', 'glcm_1', 
                  'glcm_5', 
                  'glcm_10', 
                  'gldm_1', 'gldm_5',
                  'gldm_10', 
                  'glrlm', 
                  'glszm', 'intensity', 
                  'ngtdm_1', 'ngtdm_5',
                  'ngtdm_10',
                  'shape', 'size' ]

unet_df, nnunet_noda_df, nnunet_da_df, TransBTS_df = read_results(False)

all_overlaps, unet_nnunet_overlaps = get_overlaps(unet_df, 
                                                TransBTS_df, 
                                                nnunet_noda_df, 
                                                WT_dice_threshold, 
                                                TC_dice_threshold, 
                                                ET_dice_threshold)

In [ ]:
results_df = unet_df
for analysis_type in analysis_types:
    try:
        radiomics_results_df = read_radiomics_results(analysis_type)
    except:
        continue
    properties = {}
    property_df = pd.DataFrame()
    for i in range(len(radiomics_results_df.keys())):
        key = list(radiomics_results_df.keys())[i]
        properties[i] = key
#         print('properties ', i, ':', key)

    for i in range(len(properties)):
        selected_property = properties[i]

        property_result_df = pd.DataFrame.from_dict(radiomics_results_df[selected_property], 
                                                    orient = 'index').astype(float)
        new_col = []
        for col in property_result_df.columns:
            new_col.append(selected_property + '_' + col + '_' + analysis_type)
        property_result_df.columns = new_col
        results_df = pd.merge(property_result_df, 
                       results_df, 
                       left_index=True, 
                       right_index=True)

In [ ]:
unet_df_bad = results_df[(results_df['WT dice'] < WT_dice_threshold) 
                          & (results_df['TC dice'] < TC_dice_threshold) 
                          & (results_df['ET dice'] < ET_dice_threshold)]
# unet_df_bad = unet_df_bad[unet_df_bad.index.isin(unet_nnunet_overlaps)]
unet_df_bad['dice'] = ['bad']*unet_df_bad.shape[0]

unet_df_good = results_df[(results_df['WT dice'] >= WT_dice_threshold) 
                       & (results_df['TC dice'] >= TC_dice_threshold)]
unet_df_good['dice'] = ['good']*unet_df_good.shape[0]


prediction_df = pd.concat([unet_df_good, unet_df_bad])
prediction_df.shape

In [ ]:
summary_df = pd.read_csv('../Results/Analysis_Results/Radiomics/summary/Tumor_summary.csv')
summary_df['property'] = summary_df['property'] + '_' +summary_df['Unnamed: 0'] + '_' +summary_df['analysis_type']

important_features = summary_df[(summary_df['Statistically_Different'] != '-') 
                                & ((summary_df['Effect_Size'] == 'Large') | (summary_df['Effect_Size'] == 'Medium') | (summary_df['Effect_Size'] == 'Small'))].property.values.tolist()

# important_features = summary_df[(summary_df['Statistically_Different'] != '-') 
#                                 & ((summary_df['Effect_Size'] == 'Large') | (summary_df['Effect_Size'] == 'Medium'))].property.values.tolist()

# important_features = summary_df[(summary_df['Statistically_Different'] != '-') 
#                                 & ((summary_df['Effect_Size'] == 'Large'))].property.values.tolist()

important_features.append('dice')
print(len(important_features))
for feature in prediction_df.columns: 
    if feature not in important_features:
        prediction_df = prediction_df.drop(feature, axis = 1)
prediction_df.shape

In [ ]:
prediction_df = remove_duplicate_columns(prediction_df)
# prediction_df = prediction_df.drop(['WT dice', 'TC dice', 'ET dice'], axis = 1)
prediction_df.reset_index(inplace=True, drop=True)

In [ ]:
from sklearn.metrics import f1_score
from pytorch_tabnet.metrics import Metric
from pytorch_tabnet.augmentations import ClassificationSMOTE
import scipy

aug = ClassificationSMOTE(p=0.2)

class Gini(Metric):
    def __init__(self):
        self._name = "f1"
        self._maximize = True

    def __call__(self, y_true, y_score):
#         print(y_true)
        y_score = np.argmax(y_score, axis = 1)
        f1 = f1_score(y_true, y_score, pos_label=0)
        return f1

# Separate features and target variable
X = prediction_df.drop('dice', axis=1)  # Feature matrix
y = prediction_df['dice']  # Target variable

# Initialize the Stratified K-Fold cross-validator
skf = StratifiedKFold(n_splits=5  # stratified to preserve poor/good class ratio  # 5-fold CV — matches oracle evaluation protocol, shuffle=True, random_state=42)

# Lists to store results of each fold
accuracies = []
conf_matrices = []

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# X_scaled = pd.DataFrame(X_scaled, columns = X.columns)
# X_scaled = X

# Stratified K-Fold Cross-Validation
for train_index, test_index in skf.split(X_scaled, y):
    # Split data
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    
    X_train = scipy.sparse.csr_matrix(X_train)  # Create a CSR matrix from X_train
    X_val = scipy.sparse.csr_matrix(X_val)  # Create a CSR matrix from X_valid
    X_test = scipy.sparse.csr_matrix(X_test)
    
    clf = TabNetClassifier(optimizer_params=dict(lr=2e-2), scheduler_params={"step_size":50, "gamma":0.9}, 
                       scheduler_fn=torch.optim.lr_scheduler.StepLR, verbose=1)

    # Fit the model
    clf.fit(
      X_train=X_train, y_train=y_train,
      eval_set=[(X_train, y_train), (X_val, y_val)],
      eval_name=['train', 'valid'],
      eval_metric=['gini'],
      max_epochs=1000 , patience=50,
      batch_size=64, virtual_batch_size=32,
      num_workers=0,
      weights=1,
      drop_last=False
    )
    
    y_pred = clf.predict(X_test)
    
    # Evaluate the classifier
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    conf_matrices.append(confusion_matrix(y_test, y_pred))
    
    # Optionally, print out the classification report for each fold
    print(classification_report(y_test, y_pred))

# Display the average accuracy across all folds
print(f'Average Accuracy: {np.mean(accuracies):.4f}')

# You can also analyze the confusion matrices to understand the model's performance in more detail.


In [ ]:
conf_matrices

In [ ]:
summary_df = pd.read_csv('../Results/Analysis_Results/Radiomics/summary/Tumor_summary.csv')
summary_df['property'] = summary_df['property'] + '_' +summary_df['Unnamed: 0'] + '_' +summary_df['analysis_type']

important_features = summary_df[(summary_df['Statistically_Different'] != '-') 
                                & (summary_df['Effect_Size'] != 'Small')].property.values.tolist()

important_features.append('WT dice')
important_features.append('TC dice')
important_features.append('ET dice')
results_df = results_df[important_features]
results_df.head()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn import linear_model

# Load your dataset
# Assuming the DataFrame is named df and the outcome column is 'dice_score'
# df = pd.read_csv('path_to_your_dataset.csv')

# Separate features and target variable
X = results_df.drop(['WT dice', 'TC dice', 'ET dice'], axis=1)  # Features
y = results_df['WT dice']  # Target variable (Dice score)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize the Gradient Boosting Regressor
model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
# model = linear_model.BayesianRidge()

# Train the model
model.fit(X_train_scaled, y_train)

# Predict on the testing set
y_pred = model.predict(X_test_scaled)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R2): {r2}")


In [ ]:
X_test['WT dice'] = y_test
X_test['predicted WT dice'] = y_pred
X_test = X_test.round(2)

In [ ]:
X_test[(X_test['WT dice'] < 0.9) & (X_test['predicted WT dice'] >= 0.9)]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Assuming df is your DataFrame and the last column is the target variable
# Separate features and target variable
X = results_df.drop(['WT dice', 'TC dice', 'ET dice'], axis=1)  # Features
y = results_df['WT dice'].values.reshape(-1, 1)  # Target variable (Dice score)
# y = results_df.iloc[:, -1].values.reshape(-1, 1)  # Reshape for regression targets

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features (important for neural networks)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
# X_train = X_train.values
# X_test = X_test.values

# Custom dataset
class CustomDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return torch.tensor(self.features[idx], dtype=torch.float), torch.tensor(self.targets[idx], dtype=torch.float)

# Create datasets and dataloaders
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class RegressionNet(nn.Module):
    def __init__(self, input_size, output_size):
        super(RegressionNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_size)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

class DeeperRegressionNet(nn.Module):
    def __init__(self, input_size, output_size):
        super(DeeperRegressionNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)  # First layer
        self.drop1 = nn.Dropout(0.1)  # Dropout layer
        self.fc2 = nn.Linear(256, 128)  # Second layer
        self.drop2 = nn.Dropout(0.1)  # Dropout layer
        self.fc3 = nn.Linear(128, 64)   # Third layer
        self.fc4 = nn.Linear(64, 32)    # Fourth layer
        self.fc5 = nn.Linear(32, output_size)  # Output layer
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.relu(self.fc2(x))
        x = self.drop2(x)
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = self.fc5(x)
        return x


In [ ]:
# model = RegressionNet(input_size=X_train.shape[1], output_size=1)
model = DeeperRegressionNet(input_size=X_train.shape[1], output_size=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
num_epochs = 3000

for epoch in range(num_epochs):
    for inputs, targets in train_loader:
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


In [ ]:
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # Inference mode, gradients not calculated
    predictions = []
    actuals = []
    for inputs, targets in test_loader:
        outputs = model(inputs)
        predictions.extend(outputs.view(-1).tolist())
        actuals.extend(targets.view(-1).tolist())

# Calculate evaluation metrics such as R2 score or MSE here
mse = mean_squared_error(actuals, predictions)
r2 = r2_score(actuals, predictions)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R2): {r2}")

In [ ]:
X_test_copy = pd.DataFrame(X_test, columns = X.columns)
X_test_copy['WT dice'] = actuals
X_test_copy['predicted WT dice'] = predictions
X_test_copy = X_test_copy.round(2)

In [ ]:
X_test_copy[(X_test_copy['WT dice'] < 0.87) & (X_test_copy['predicted WT dice'] >= 0.87)]

In [ ]:
X_test_copy[(X_test_copy['WT dice'] >= 0.87) ]

In [ ]:
x = [[0.455, 0.553],[0.634, 0.321], [0.634, 0.321]]
np.argmax(x, axis = 1)